# YouTube Spam Classification with Naive Bayes

## 1. Problem Statement
## 2. Dataset
## 3. Exploratory Data Analysis
## 4. Data Preparation
## 5. Baseline: sklearn MultinomialNB
## 6. Multinomial Naive Bayes from Scratch
## 7. Hyperparameter Tuning
## 8. Final Evaluation
## 9. Error Analysis
## 10. Conclusions

## Problem Statement

The goal of this project is to build a binary classifier for detecting
spam in YouTube comments.

The project focuses on understanding and implementing Multinomial Naive
Bayes from scratch and comparing the implementation with scikit-learn.


##Dataset

In [78]:
!pip install ucimlrepo -q

In [3]:
from ucimlrepo import fetch_ucirepo

youtube_spam = fetch_ucirepo(id=380)

X = youtube_spam.data.features
y = youtube_spam.data.targets

##EDA

In [6]:
import pandas as pd

In [7]:
df = pd.DataFrame({
    "text": X["CONTENT"],
    "target": y["CLASS"]
})

df.head()

,text,target
0,"Huh, anyway check out this you[tube] channel: ...",1
1,Hey guys check out my new channel and our firs...,1
2,just for test I have to say murdev.com,1
3,me shaking my sexy ass on my channel enjoy ^_^ ﻿,1
4,watch?v=vtaRGgvGtWQ Check this out .﻿,1


In [10]:
df["target"].value_counts(normalize = "True")

,proportion
target,
1,0.513804
0,0.486196


In [11]:
df.isna().sum()

,0
text,0
target,0


In [12]:
df.duplicated().sum()

np.int64(196)

In [13]:
conflicts = (df.groupby("text")["target"]).nunique()
conflicts[conflicts > 1]

,target
text,


In [14]:
df = df.drop_duplicates().reset_index(drop = True)

196 duplicated observations were found.
Duplicate comments were removed before splitting the dataset to prevent
the same text from appearing in both train and test sets.

##Data prep

```
# Выбран кодовый формат
```



In [15]:
X = df["text"]
y = df["target"]

In [16]:
from sklearn.model_selection import train_test_split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [18]:
from sklearn.feature_extraction.text import CountVectorizer

In [21]:
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [22]:
vectorizer.get_feature_names_out()[:30]

array(['00', '000', '002', '018', '034', '047000', '053012', '08', '09',
       '0d878a889c', '0laviqu2b', '10', '100', '1000', '10000000',
       '1000000000', '100000415527985', '100007085325116', '10001',
       '100877300245414', '101721377578919894134', '10200253113705769',
       '1030', '104999962146104962510', '10626835', '107297364',
       '1073741828', '1073741830', '1073741943', '108k'], dtype=object)

In [23]:
X_train.iloc[0]

'Love you shkira\ufeff'

In [25]:
X_train_vec[0].toarray()

array([[0, 0, 0, ..., 0, 0, 0]])

##Baseline Model


As a baseline, scikit-learn's `MultinomialNB` was trained using a
Bag-of-Words representation produced by `CountVectorizer`.

In [26]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

In [27]:
y_pred = model.predict(X_test_vec)

In [28]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.91      0.94       184
           1       0.91      0.97      0.94       168

    accuracy                           0.94       352
   macro avg       0.94      0.94      0.94       352
weighted avg       0.94      0.94      0.94       352



##Multinomial Naive Bayes from Scratch

For each class, the model estimates:

- class prior P(c)
- conditional probabilities P(word | c)

Laplace smoothing is used to avoid zero probabilities.

Predictions are computed as logarithms to avoid numerical underflow using simple formula which comes from taking a log of our naive representation of a $P(c|x)$:
$$
\log P(c|x)
∝
\log P(c)
+
\sum_j x_j \log P(w_j|c)
$$

**Simple case**

In [31]:
class_counts = y_train.value_counts()
class_counts

,count
target,
0,735
1,673


In [32]:
class_priors = class_counts / len(y_train)

class_priors

,count
target,
0,0.522017
1,0.477983


In [33]:
ham_mask = y_train.to_numpy() == 0
spam_mask = y_train.to_numpy() == 1

In [34]:
ham_word_counts = X_train_vec[ham_mask].sum(axis=0)
spam_word_counts = X_train_vec[spam_mask].sum(axis=0)

In [35]:
ham_word_counts.shape, spam_word_counts.shape

((1, 3851), (1, 3851))

In [36]:
import numpy as np

ham_word_counts = np.asarray(ham_word_counts).ravel()
spam_word_counts = np.asarray(spam_word_counts).ravel()

In [37]:
words = vectorizer.get_feature_names_out()
top_spam = np.argsort(spam_word_counts)[-10:][::-1]

In [42]:
alpha = 1

vocab_size = len(words)

ham_word_probs = (ham_word_counts + alpha) / (ham_word_counts.sum() + alpha * vocab_size)

spam_word_probs = (spam_word_counts + alpha) / (spam_word_counts.sum() + alpha * vocab_size)

In [44]:
ham_word_probs.sum(), spam_word_probs.sum()

(np.float64(0.9999999999999999), np.float64(1.0))

In [45]:
text = X_test.iloc[0]
x = X_test_vec[0]

text

'OH SHIT THIS WAS UPLOADED ON MY BIRTHDAY, finally my birthday isnt a curse, because on this day maylaysia airlines went missing, so i thought my birthday was a curse xD\ufeff'

In [47]:
ham_score = (np.log(class_priors[0]) + x @ np.log(ham_word_probs))
spam_score = (np.log(class_priors[1]) + x @ np.log(spam_word_probs))

ham_score, spam_score

(array([-163.90833341]), array([-171.84834962]))

In [48]:
prediction = 0 if ham_score > spam_score else 1
prediction

0

In [49]:
word_indices = x.indices
word_counts = x.data

In [50]:
for idx, count in zip(word_indices, word_counts):
  print(words[idx], "count =", count, "P(word|ham) =", ham_word_probs[idx], "P(word|spam) =", spam_word_probs[idx])

because count = 1 P(word|ham) = 0.0021313698895562877 P(word|spam) = 0.0010070493454179255
birthday count = 3 P(word|ham) = 0.0005812826971517147 P(word|spam) = 0.00025176233635448137
day count = 1 P(word|ham) = 0.0006781631466770006 P(word|spam) = 0.0007552870090634441
finally count = 1 P(word|ham) = 0.00019376089905057158 P(word|spam) = 0.00010070493454179255
isnt count = 1 P(word|ham) = 0.00019376089905057158 P(word|spam) = 0.00010070493454179255
missing count = 1 P(word|ham) = 9.688044952528579e-05 P(word|spam) = 0.00010070493454179255
my count = 3 P(word|ham) = 0.0034876961829102887 P(word|spam) = 0.017623363544813697
oh count = 1 P(word|ham) = 0.00019376089905057158 P(word|spam) = 0.00015105740181268882
on count = 2 P(word|ham) = 0.0035845766324355743 P(word|spam) = 0.008408862034239678
shit count = 1 P(word|ham) = 0.0011625653943034295 P(word|spam) = 0.0004028197381671702
so count = 1 P(word|ham) = 0.006975392365820577 P(word|spam) = 0.004078549848942598
this count = 2 P(word|ha

**Class realisation**

In [56]:
class MultinomialNaiveBayes:
  def __init__(self, alpha = 1.0):
    self.alpha = alpha

  def fit(self, X, y):
    y = np.asarray(y)

    self.classes_ = np.unique(y)
    n_samples, n_features = X.shape
    self.class_log_prior_ = np.zeros(len(self.classes_))
    self.feature_log_prob_ = np.zeros((len(self.classes_), n_features))

    for i, cls in enumerate(self.classes_):
      X_cls = X[y == cls]

      self.class_log_prior_[i] = np.log(X_cls.shape[0] / n_samples)

      word_counts = np.asarray(X_cls.sum(axis=0)).ravel()
      smoothed_counts = word_counts + self.alpha

      word_probs = (smoothed_counts / smoothed_counts.sum())
      self.feature_log_prob_[i] = np.log(word_probs)

    return self


  def predict(self, X):

    scores = X @ self.feature_log_prob_.T + self.class_log_prior_

    class_indices = np.asarray(scores).argmax(axis=1)

    return self.classes_[class_indices]


In [57]:
my_model = MultinomialNaiveBayes(alpha = 1.0)
my_model.fit(X_train_vec, y_train)

In [59]:
y_pred_custom = my_model.predict(X_test_vec)

In [60]:
print("Custom:")
print(classification_report(y_test, y_pred_custom))

print("Sklearn:")
print(classification_report(y_test, y_pred_custom))

Custom:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94       184
           1       0.91      0.97      0.94       168

    accuracy                           0.94       352
   macro avg       0.94      0.94      0.94       352
weighted avg       0.94      0.94      0.94       352

Sklearn:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94       184
           1       0.91      0.97      0.94       168

    accuracy                           0.94       352
   macro avg       0.94      0.94      0.94       352
weighted avg       0.94      0.94      0.94       352



The custom implementation produced exactly the same predictions as
scikit-learn's MultinomialNB with the same hyperparameters.

##Hyperparameter tuning

The smoothing parameter $\alpha$ will be selected using the validation set.

In [62]:
from sklearn.model_selection import train_test_split

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [63]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_val = CountVectorizer()

X_train_sub_vec = vectorizer_val.fit_transform(X_train_sub)
X_val_vec = vectorizer_val.transform(X_val)

In [64]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

for alpha in [0.01, 0.1, 0.5, 1, 2, 5, 10]:
    model = MultinomialNaiveBayes(alpha=alpha)

    model.fit(X_train_sub_vec, y_train_sub)

    y_val_pred = model.predict(X_val_vec)

    results.append({
        "alpha": alpha,
        "accuracy": accuracy_score(y_val, y_val_pred),
        "precision_spam": precision_score(y_val, y_val_pred),
        "recall_spam": recall_score(y_val, y_val_pred),
        "f1_spam": f1_score(y_val, y_val_pred)
    })

In [65]:
results_df = pd.DataFrame(results)

results_df

,alpha,accuracy,precision_spam,recall_spam,f1_spam
0,0.01,0.925532,0.919118,0.925926,0.922509
1,0.10,0.918440,0.924242,0.903704,0.913858
2,0.50,0.932624,0.932836,0.925926,0.929368
3,1.00,0.932624,0.926471,0.933333,0.929889
4,2.00,0.932624,0.902778,0.962963,0.931900
5,5.00,0.886525,0.819876,0.977778,0.891892
6,10.00,0.858156,0.774566,0.992593,0.870130


In [66]:
results_df.sort_values(
    "f1_spam",
    ascending=False
)

,alpha,accuracy,precision_spam,recall_spam,f1_spam
4,2.00,0.932624,0.902778,0.962963,0.931900
3,1.00,0.932624,0.926471,0.933333,0.929889
2,0.50,0.932624,0.932836,0.925926,0.929368
0,0.01,0.925532,0.919118,0.925926,0.922509
1,0.10,0.918440,0.924242,0.903704,0.913858
5,5.00,0.886525,0.819876,0.977778,0.891892
6,10.00,0.858156,0.774566,0.992593,0.870130


The best validation F1-score for the spam class was obtained with
$\alpha = 2$.

##Final model

In [67]:
final_model = MultinomialNaiveBayes(alpha = 2)
final_model.fit(X_train_vec, y_train)

y_pred_final = final_model.predict(X_test_vec)

In [68]:
print(classification_report(y_test, y_pred_final, digits=3))

              precision    recall  f1-score   support

           0      0.976     0.897     0.935       184
           1      0.896     0.976     0.934       168

    accuracy                          0.935       352
   macro avg      0.936     0.936     0.935       352
weighted avg      0.938     0.935     0.935       352



In [69]:
errors = pd.DataFrame({"text": X_test.reset_index(drop=True), "true": y_test.reset_index(drop=True), "pred": y_pred_final})
errors.head()

,text,true,pred
0,"OH SHIT THIS WAS UPLOADED ON MY BIRTHDAY, fina...",0,0
1,Hi everyone! Do you like music? Then why not c...,1,1
2,Love this song so much! One of my faves! Xxx﻿,0,0
3,Hi I&#39;m lil m !!! Check out love the way yo...,1,1
4,I love this song because we sing it at Camp al...,0,0


The final model achieved a spam recall of 97.6%, detecting 164 of 168
spam comments in the test set.

The model produced 19 false positives and only 4 false negatives.

##Error Analysis

In [71]:
false_positive = errors[(errors["true"] == 0) & (errors["pred"] == 1)]
false_positive

,text,true,pred
23,Im just to check how much views it has﻿,0,1
33,My telephone!﻿,0,1
84,"blue eyes, can't be trusted. uranus bless ame...",0,1
118,"WE GO FOR 1,000,000,000 FOR EMINEM﻿",0,1
139,"You exactly who u want to be,watching your fav...",0,1
140,Katy has conqueror's haki &gt;:)﻿,0,1
141,So he&#39;s admitting he killed his girlfriend...,0,1
151,everyday I&#39;m shufflin﻿,0,1
196,I like it<br />﻿,0,1
202,Lemme Top Comments Please!!﻿,0,1


In [72]:
false_negative = errors[(errors["true"] == 1) & (errors["pred"] == 0)]
false_negative

,text,true,pred
129,Share this video.. This song can beat PSY - Ga...,1,0
167,subscribed :) btw you have a good style keep i...,1,0
334,this song is so addicting. the hook is dope an...,1,0
351,Fuck it was the best ever 0687119038 nummber o...,1,0


In [75]:
ham_idx = np.where(final_model.classes_ == 0)[0][0]
spam_idx = np.where(final_model.classes_ == 1)[0][0]

log_ratio = (final_model.feature_log_prob_[spam_idx] - final_model.feature_log_prob_[ham_idx])

In [76]:
top_spam_idx = np.argsort(log_ratio)[-20:][::-1]

pd.DataFrame({"word": vectorizer.get_feature_names_out()[top_spam_idx],"log_ratio": log_ratio[top_spam_idx]})

,word,log_ratio
0,subscribe,3.557271
1,channel,3.397423
2,please,3.361795
3,com,3.314041
4,https,3.234904
5,facebook,2.724078
6,check,2.617544
7,out,2.573190
8,www,2.500935
9,visit,2.455814


As we can see, our model missed a spam message containing the word "subscribed", even though the word "subscribe" is considered highly suspicious. This reveals an important limitation of our model: different forms of the same word are treated as separate features.  
On the other hand, we can see that "zonepa" is also one of the most suspicious words. It turns out that our model can assign high spam scores to rarely occurring words, especially when they mostly appear in spam messages.

##Conclusion

Error analysis revealed several limitations of the Bag-of-Words
Naive Bayes approach:

- morphological variants are treated as independent tokens
  (`subscribe` vs `subscribed`);
- the model does not understand context;
- rare tokens may receive disproportionately large likelihood ratios;
- patterns such as phone numbers are treated as individual tokens rather
  than as a general concept.